In [1]:
import pandas as pd
from pathlib import Path

STORE_DF_PATH = Path("../data/raw/store.csv")
TRAIN_DF_PATH = Path("../data/raw/train.csv")

store_df = pd.read_csv(STORE_DF_PATH)
train_df = pd.read_csv(TRAIN_DF_PATH, dtype={
    "StateHoliday": "category"
})

# Store Dataframe

In [2]:
# look at the store dataframe
store_df.head()

,Store,StoreType,Assortment,CompetitionDistance,CompetitionOpenSinceMonth,CompetitionOpenSinceYear,Promo2,Promo2SinceWeek,Promo2SinceYear,PromoInterval
0,1,c,a,1270.0,9.0,2008.0,0,NaN,NaN,NaN
1,2,a,a,570.0,11.0,2007.0,1,13.0,2010.0,"Jan,Apr,Jul,Oct"
2,3,a,a,14130.0,12.0,2006.0,1,14.0,2011.0,"Jan,Apr,Jul,Oct"
3,4,c,c,620.0,9.0,2009.0,0,NaN,NaN,NaN
4,5,a,a,29910.0,4.0,2015.0,0,NaN,NaN,NaN


In [3]:
# shape
store_df.shape

(1115, 10)

In [4]:
# info about the df
store_df.info(memory_usage="deep")

<class 'pandas.DataFrame'>
RangeIndex: 1115 entries, 0 to 1114
Data columns (total 10 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   Store                      1115 non-null   int64  
 1   StoreType                  1115 non-null   str    
 2   Assortment                 1115 non-null   str    
 3   CompetitionDistance        1112 non-null   float64
 4   CompetitionOpenSinceMonth  761 non-null    float64
 5   CompetitionOpenSinceYear   761 non-null    float64
 6   Promo2                     1115 non-null   int64  
 7   Promo2SinceWeek            571 non-null    float64
 8   Promo2SinceYear            571 non-null    float64
 9   PromoInterval              571 non-null    str    
dtypes: float64(5), int64(2), str(3)
memory usage: 98.0 KB


In [5]:
# check duplicates
store_df.duplicated().sum()

np.int64(0)

In [6]:
# we can change some str columns to category for better performance
store_df.StoreType = store_df.StoreType.astype("category")
store_df.Assortment = store_df.Assortment.astype("category")

In [7]:
# check NaN values
store_df.isna().sum()

Store                          0
StoreType                      0
Assortment                     0
CompetitionDistance            3
CompetitionOpenSinceMonth    354
CompetitionOpenSinceYear     354
Promo2                         0
Promo2SinceWeek              544
Promo2SinceYear              544
PromoInterval                544
dtype: int64

In [8]:
# fill CompetitionDistance with median
store_df.CompetitionDistance = store_df.CompetitionDistance.fillna(store_df.CompetitionDistance.median()).astype("int32")

# fill other competition-related features with 0
comp_cols = ["CompetitionOpenSinceMonth", "CompetitionOpenSinceYear"]

for col in comp_cols:
    store_df[col] = store_df[col].fillna(0).astype("int16")

In [9]:
# fill Promo2SinceWeek and Promo2SinceYear columns with 0 and PromoInterval with None
store_df.Promo2SinceWeek = store_df.Promo2SinceWeek.fillna(0).astype("int8")
store_df.Promo2SinceYear = store_df.Promo2SinceYear.fillna(0).astype("int16")
store_df.PromoInterval = store_df.PromoInterval.fillna("None").astype("category")

In [10]:
# check that Promo2=0 stores have week 0, year 0 and interval None
store_df.loc[store_df.Promo2 == 0, ["Promo2SinceWeek", "Promo2SinceYear", "PromoInterval"]].head()

,Promo2SinceWeek,Promo2SinceYear,PromoInterval
0,0,0,None
3,0,0,None
4,0,0,None
5,0,0,None
6,0,0,None


In [11]:
# check that there are no negative values for competition and promotion features
for col in store_df.select_dtypes(include="number"):
    print(f"{store_df[col].name} values below 0 count: {(store_df[col] < 0).sum()}")

Store values below 0 count: 0
CompetitionDistance values below 0 count: 0
CompetitionOpenSinceMonth values below 0 count: 0
CompetitionOpenSinceYear values below 0 count: 0
Promo2 values below 0 count: 0
Promo2SinceWeek values below 0 count: 0
Promo2SinceYear values below 0 count: 0


In [12]:
# check NaN values
store_df.isna().sum()

Store                        0
StoreType                    0
Assortment                   0
CompetitionDistance          0
CompetitionOpenSinceMonth    0
CompetitionOpenSinceYear     0
Promo2                       0
Promo2SinceWeek              0
Promo2SinceYear              0
PromoInterval                0
dtype: int64

In [13]:
# info
store_df.info(memory_usage="deep")

<class 'pandas.DataFrame'>
RangeIndex: 1115 entries, 0 to 1114
Data columns (total 10 columns):
 #   Column                     Non-Null Count  Dtype   
---  ------                     --------------  -----   
 0   Store                      1115 non-null   int64   
 1   StoreType                  1115 non-null   category
 2   Assortment                 1115 non-null   category
 3   CompetitionDistance        1115 non-null   int32   
 4   CompetitionOpenSinceMonth  1115 non-null   int16   
 5   CompetitionOpenSinceYear   1115 non-null   int16   
 6   Promo2                     1115 non-null   int64   
 7   Promo2SinceWeek            1115 non-null   int8    
 8   Promo2SinceYear            1115 non-null   int16   
 9   PromoInterval              1115 non-null   category
dtypes: category(3), int16(3), int32(1), int64(2), int8(1)
memory usage: 33.1 KB


In [14]:
# check final dataframe
store_df.head()

,Store,StoreType,Assortment,CompetitionDistance,CompetitionOpenSinceMonth,CompetitionOpenSinceYear,Promo2,Promo2SinceWeek,Promo2SinceYear,PromoInterval
0,1,c,a,1270,9,2008,0,0,0,None
1,2,a,a,570,11,2007,1,13,2010,"Jan,Apr,Jul,Oct"
2,3,a,a,14130,12,2006,1,14,2011,"Jan,Apr,Jul,Oct"
3,4,c,c,620,9,2009,0,0,0,None
4,5,a,a,29910,4,2015,0,0,0,None


# Train Dataframe

In [15]:
# look at the train dataframe
train_df.head()

,Store,DayOfWeek,Date,Sales,Customers,Open,Promo,StateHoliday,SchoolHoliday
0,1,5,2015-07-31,5263,555,1,1,0,1
1,2,5,2015-07-31,6064,625,1,1,0,1
2,3,5,2015-07-31,8314,821,1,1,0,1
3,4,5,2015-07-31,13995,1498,1,1,0,1
4,5,5,2015-07-31,4822,559,1,1,0,1


In [16]:
# shape
train_df.shape

(1017209, 9)

In [17]:
# info
train_df.info(memory_usage="deep")

<class 'pandas.DataFrame'>
RangeIndex: 1017209 entries, 0 to 1017208
Data columns (total 9 columns):
 #   Column         Non-Null Count    Dtype   
---  ------         --------------    -----   
 0   Store          1017209 non-null  int64   
 1   DayOfWeek      1017209 non-null  int64   
 2   Date           1017209 non-null  str     
 3   Sales          1017209 non-null  int64   
 4   Customers      1017209 non-null  int64   
 5   Open           1017209 non-null  int64   
 6   Promo          1017209 non-null  int64   
 7   StateHoliday   1017209 non-null  category
 8   SchoolHoliday  1017209 non-null  int64   
dtypes: category(1), int64(7), str(1)
memory usage: 72.8 MB


In [18]:
# check duplicate values
train_df.duplicated().sum()

np.int64(0)

In [19]:
# check NaN values
train_df.isna().sum()

Store            0
DayOfWeek        0
Date             0
Sales            0
Customers        0
Open             0
Promo            0
StateHoliday     0
SchoolHoliday    0
dtype: int64

In [20]:
# change Date to date type
train_df.Date = pd.to_datetime(train_df.Date, errors="coerce")

In [21]:
# check min. and max. values to optimize dataset performance
for col in train_df.select_dtypes(include="number").columns:
    print(f"{train_df[col].name} Min.: {train_df[col].min()} | Max. {train_df[col].max()}")

Store Min.: 1 | Max. 1115
DayOfWeek Min.: 1 | Max. 7
Sales Min.: 0 | Max. 41551
Customers Min.: 0 | Max. 7388
Open Min.: 0 | Max. 1
Promo Min.: 0 | Max. 1
SchoolHoliday Min.: 0 | Max. 1


In [22]:
# change data types
train_df["Store"] = train_df["Store"].astype("int16")
train_df["Customers"] = train_df["Customers"].astype("int16")
train_df["Sales"] = train_df["Sales"].astype("int32")

train_df[["DayOfWeek", "Open", "Promo", "SchoolHoliday"]] = train_df[["DayOfWeek", "Open", "Promo", "SchoolHoliday"]].astype("int8")

In [23]:
# check StateHoliday unique values
train_df.StateHoliday.unique()

['0', 'a', 'b', 'c']
Categories (4, str): ['0', 'a', 'b', 'c']

In [24]:
# merge the two dataframes
merged_df = train_df.merge(store_df, how="left", on="Store")

In [25]:
# move the target (Sales to the end)
col_to_move = merged_df.pop("Sales")
merged_df.insert(len(merged_df.columns), "Sales", col_to_move)

In [26]:
# check NaN values after merge
assert merged_df.isna().sum().sum() == 0

In [27]:
# merged df info
merged_df.info(memory_usage="deep")

<class 'pandas.DataFrame'>
RangeIndex: 1017209 entries, 0 to 1017208
Data columns (total 18 columns):
 #   Column                     Non-Null Count    Dtype         
---  ------                     --------------    -----         
 0   Store                      1017209 non-null  int16         
 1   DayOfWeek                  1017209 non-null  int8          
 2   Date                       1017209 non-null  datetime64[us]
 3   Customers                  1017209 non-null  int16         
 4   Open                       1017209 non-null  int8          
 5   Promo                      1017209 non-null  int8          
 6   StateHoliday               1017209 non-null  category      
 7   SchoolHoliday              1017209 non-null  int8          
 8   StoreType                  1017209 non-null  category      
 9   Assortment                 1017209 non-null  category      
 10  CompetitionDistance        1017209 non-null  int32         
 11  CompetitionOpenSinceMonth  1017209 non-null  int

In [28]:
# check the final dataframe
merged_df.head()

,Store,DayOfWeek,Date,Customers,Open,Promo,StateHoliday,SchoolHoliday,StoreType,Assortment,CompetitionDistance,CompetitionOpenSinceMonth,CompetitionOpenSinceYear,Promo2,Promo2SinceWeek,Promo2SinceYear,PromoInterval,Sales
0,1,5,2015-07-31,555,1,1,0,1,c,a,1270,9,2008,0,0,0,None,5263
1,2,5,2015-07-31,625,1,1,0,1,a,a,570,11,2007,1,13,2010,"Jan,Apr,Jul,Oct",6064
2,3,5,2015-07-31,821,1,1,0,1,a,a,14130,12,2006,1,14,2011,"Jan,Apr,Jul,Oct",8314
3,4,5,2015-07-31,1498,1,1,0,1,c,c,620,9,2009,0,0,0,None,13995
4,5,5,2015-07-31,559,1,1,0,1,a,a,29910,4,2015,0,0,0,None,4822


In [29]:
# save it
merged_df.to_parquet("../data/processed/merged_df.parquet", index=False)